# Dealing with tabular data

### Created by Wei Jin and Zewen Liu

Please feel free to reach out with questions or suggestions: wei.jin@emory.edu and zewen.liu@emory.edu

Let's use an example from Kaggle: https://www.kaggle.com/datasets/meirnizri/covid19-dataset

The final classification goal is covid test findings.
Values 1-3 mean that the patient was diagnosed with covid in different degrees. 4 or higher means that the patient is not a carrier of covid or that the test is inconclusive.

In [ ]:
import kagglehub

# Download example dataset
path = kagglehub.dataset_download("meirnizri/covid19-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/covid19-dataset


In [ ]:
import os
import pandas as pd
data = pd.read_csv(os.path.join(path, "Covid Data.csv"))
print(f"length: {len(data)}")
data.head()

length: 1048575


,USMER,MEDICAL_UNIT,SEX,PATIENT_TYPE,DATE_DIED,INTUBED,PNEUMONIA,AGE,PREGNANT,DIABETES,...,ASTHMA,INMSUPR,HIPERTENSION,OTHER_DISEASE,CARDIOVASCULAR,OBESITY,RENAL_CHRONIC,TOBACCO,CLASIFFICATION_FINAL,ICU
0,2,1,1,1,03/05/2020,97,1,65,2,2,...,2,2,1,2,2,2,2,2,3,97
1,2,1,2,1,03/06/2020,97,1,72,97,2,...,2,2,1,2,2,1,1,2,5,97
2,2,1,2,2,09/06/2020,1,2,55,97,1,...,2,2,2,2,2,2,2,2,3,2
3,2,1,1,1,12/06/2020,97,2,53,2,2,...,2,2,2,2,2,2,2,2,7,97
4,2,1,2,1,21/06/2020,97,2,68,97,1,...,2,2,1,2,2,2,2,2,3,97


In [ ]:
# select a subset of dataset
data = data.iloc[:10000]

In [ ]:
# check variable summary
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   USMER                 10000 non-null  int64 
 1   MEDICAL_UNIT          10000 non-null  int64 
 2   SEX                   10000 non-null  int64 
 3   PATIENT_TYPE          10000 non-null  int64 
 4   DATE_DIED             10000 non-null  object
 5   INTUBED               10000 non-null  int64 
 6   PNEUMONIA             10000 non-null  int64 
 7   AGE                   10000 non-null  int64 
 8   PREGNANT              10000 non-null  int64 
 9   DIABETES              10000 non-null  int64 
 10  COPD                  10000 non-null  int64 
 11  ASTHMA                10000 non-null  int64 
 12  INMSUPR               10000 non-null  int64 
 13  HIPERTENSION          10000 non-null  int64 
 14  OTHER_DISEASE         10000 non-null  int64 
 15  CARDIOVASCULAR        10000 non-null 

In [ ]:
# drop columns
data.drop(columns=['DATE_DIED'],inplace=True)
# features
features = data.drop(columns=['CLASIFFICATION_FINAL'])
# target
target = data['CLASIFFICATION_FINAL']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder

# Convert target to binary (1-3: Covid, 4+: No Covid)
le = LabelEncoder()
binary_target = le.fit_transform(target.apply(lambda x: 1 if x >= 1 and x <= 3 else 0))

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, binary_target, test_size=0.2, random_state=42)

In [ ]:
from sklearn.neural_network import MLPClassifier

nn_model = MLPClassifier(random_state=42, max_iter=1000) # Increased max_iter for convergence
print("Training MLP model...")
nn_model.fit(X_train, y_train)

Training MLP model...


MLPClassifier(max_iter=1000, random_state=42)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# Evaluate MLP model
print("\nEvaluating MLP model on the test set...")
nn_predictions = nn_model.predict(X_test)
nn_accuracy = accuracy_score(y_test, nn_predictions)
nn_precision = precision_score(y_test, nn_predictions)
nn_recall = recall_score(y_test, nn_predictions)
nn_f1 = f1_score(y_test, nn_predictions)

print(f"  Accuracy: {nn_accuracy:.4f}")
print(f"  Precision: {nn_precision:.4f}")
print(f"  Recall: {nn_recall:.4f}")
print(f"  F1 Score: {nn_f1:.4f}")


Evaluating MLP model on the test set...
  Accuracy: 0.8905
  Precision: 0.8966
  Recall: 0.9885
  F1 Score: 0.9403


In [ ]:
from xgboost import XGBClassifier

# Initialize models
xgb_model = XGBClassifier(random_state=42)

# Train models
print("Training XGBoost model...")
xgb_model.fit(X_train, y_train)

# Evaluate XGBoost model
print("Evaluating XGBoost model on the test set...")
xgb_predictions = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test, xgb_predictions)
xgb_precision = precision_score(y_test, xgb_predictions)
xgb_recall = recall_score(y_test, xgb_predictions)
xgb_f1 = f1_score(y_test, xgb_predictions)

print(f"  Accuracy: {xgb_accuracy:.4f}")
print(f"  Precision: {xgb_precision:.4f}")
print(f"  Recall: {xgb_recall:.4f}")
print(f"  F1 Score: {xgb_f1:.4f}")

Training XGBoost model...
Evaluating XGBoost model on the test set...
  Accuracy: 0.8815
  Precision: 0.8940
  Recall: 0.9805
  F1 Score: 0.9353


## In-Class Quiz
Can you complete the following code block to use a different model to train on your dataset and evaluate it?

You may simply use `RandomForestClassifier()` which will set all hyperparameters to default values.

Documentation: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html


Popular models you can try: RandomForestClassifier, MLPClassifier, XGBOOST, SVM

(the tree-based models are also popular like GradientBoostingClassifier)


In [14]:
from sklearn.ensemble import RandomForestClassifier

#### Can you complete this block of code? ####
##############################################
##############################################
############YOUR CODE BLOCK###################
##############################################
##############################################
##############################################
##############################################

# nn_model = RandomForestClassifier(n_estimators=200) # Increased max_iter for convergence
# print("Training RandomForestClassifier model...")
# nn_model.fit(X_train, y_train)

Training RandomForestClassifier model...


RandomForestClassifier(n_estimators=200)

In [15]:
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# # Evaluate RandomForestClassifier model
# print("\nEvaluating RandomForestClassifier model on the test set...")
# nn_predictions = nn_model.predict(X_test)
# nn_accuracy = accuracy_score(y_test, nn_predictions)
# nn_precision = precision_score(y_test, nn_predictions)
# nn_recall = recall_score(y_test, nn_predictions)
# nn_f1 = f1_score(y_test, nn_predictions)

# print(f"  Accuracy: {nn_accuracy:.4f}")
# print(f"  Precision: {nn_precision:.4f}")
# print(f"  Recall: {nn_recall:.4f}")
# print(f"  F1 Score: {nn_f1:.4f}")


Evaluating RandomForestClassifier model on the test set...
  Accuracy: 0.8630
  Precision: 0.8936
  Recall: 0.9570
  F1 Score: 0.9242


### include hyperparameter tuning

In [16]:
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

xgb_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5, 7]
}

nn_param_grid = {
    'hidden_layer_sizes': [(64, 32), (128, 64)],
    'alpha': [0.0001, 0.001],
    'learning_rate_init': [0.001, 0.01]
}

In [17]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define a function to perform hyperparameter tuning with cross-validation
def tune_model_with_cross_validation(model, param_grid, X_train, y_train, X_test, y_test, cv=5, scoring='accuracy'):
    """
    Tunes a given model using GridSearchCV with cross-validation and evaluates the best model.

    Args:
        model: The machine learning model to tune.
        param_grid: The hyperparameter grid to search over.
        X_train: The training data features.
        y_train: The training data target.
        X_test: The testing data features.
        y_test: The testing data target.
        cv: The number of cross-validation folds.
        scoring: The scoring metric to use for evaluation.

    Returns:
        A dictionary containing the evaluation metrics and best parameters of the tuned model.
    """
    print(f"Performing GridSearchCV for {type(model).__name__}...")
    # Initialize GridSearchCV with the model, parameter grid, cross-validation folds, and scoring metric
    grid_search = GridSearchCV(model, param_grid, cv=cv, scoring=scoring)
    # Fit GridSearchCV to the training data to find the best parameters
    grid_search.fit(X_train, y_train)
    # Get the best model found by GridSearchCV
    best_model = grid_search.best_estimator_
    print(f"Best parameters for {type(model).__name__}:", grid_search.best_params_)

    # Evaluate the best model on the test data
    print(f"Evaluating the best {type(model).__name__} model on the test set...")
    # Make predictions on the test set using the best model
    predictions = best_model.predict(X_test)
    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)

    # Print the evaluation metrics
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1 Score: {f1:.4f}")

    # Return a dictionary of the evaluation results and best parameters
    return {
        'model': type(model).__name__,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'best_params': grid_search.best_params_
    }

In [18]:
# Example usage with Random Forest model
rf_model = RandomForestClassifier()
rf_evaluation = tune_model_with_cross_validation(rf_model, rf_param_grid, X_train, y_train, X_test, y_test)
print("\nRandom Forest Evaluation Results:")
print(rf_evaluation)

Performing GridSearchCV for RandomForestClassifier...
Best parameters for RandomForestClassifier: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}
Evaluating the best RandomForestClassifier model on the test set...
  Accuracy: 0.8915
  Precision: 0.8967
  Recall: 0.9897
  F1 Score: 0.9409

Random Forest Evaluation Results:
{'model': 'RandomForestClassifier', 'accuracy': 0.8915, 'precision': 0.8967306694343539, 'recall': 0.9896907216494846, 'f1_score': 0.9409202286958889, 'best_params': {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}}


## 🔍Classification Models from scikit-learn

### 🔹 Tree-Based Models
- **RandomForestClassifier**
- **GradientBoostingClassifier**
- **HistGradientBoostingClassifier**
- **ExtraTreesClassifier**
- **XGBClassifier**

---

### 🔹 Neural Network Models
- **MLPClassifier**

---

### 🔹 Linear Models
- **LogisticRegression**
- **RidgeClassifier**
- **SGDClassifier**

---

### 🔹 Support Vector Machines
- **SVC**
- **LinearSVC**

---

### 🔹 Naive Bayes
- **GaussianNB**
- **MultinomialNB**
- **BernoulliNB**

---

### 🔹 K-Nearest Neighbors
- **KNeighborsClassifier**

---

### 🔹 Ensemble Meta-Models
- **VotingClassifier**
- **StackingClassifier**
- **BaggingClassifier**

## Homework
1. Try other classification models defined in sklearn.
2. Include hyperparameter tuning for at least one model.